Este programa utiliza la librería Scikit-learn.
Asegúrese de tenerla instalada

**Inicio del código de entrenamiento**

In [ ]:
# Paso 1: Importar librerias
import pandas as pd
import numpy as np
import os
import joblib

from google.colab import drive

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
# Paso 2: Montar Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).




Ahora que los datos están cargados y unificados, los prepararemos para el aprendizaje automático separando las características (datos de píxeles) de la etiqueta objetivo. Luego, dividiremos el conjunto de datos en conjuntos de entrenamiento y prueba.

In [ ]:
# Paso 3: Preparar los Datos para el entrenamiento
# Definir rutas de entrada y salida
input_folder = '/content/drive/MyDrive/FotosIAproyecto/DatasetGrupal'
output_folder = '/content/drive/MyDrive/FotosIAproyecto/Modelo'

# Construir la ruta completa al archivo CSV
ruta_dataset = os.path.join(input_folder, 'dataset_final_unificado.csv')

# Cargar el dataset
try:
    datasetgrupal_total = pd.read_csv(ruta_dataset)
    print(f"Dataset '{ruta_dataset}' cargado exitosamente. Primeras 5 filas:\n{datasetgrupal_total.head()}")
except FileNotFoundError:
    print(f"Error: El archivo '{ruta_dataset}' no fue encontrado. Por favor, verifica la ruta y el nombre del archivo.")
except Exception as e:
    print(f"Ocurrió un error al cargar el dataset: {e}")

Dataset '/content/drive/MyDrive/FotosIAproyecto/DatasetGrupal/dataset_final_unificado.csv' cargado exitosamente. Primeras 5 filas:
   1  1.1  1.2  1.3  1.4  1.5  1.6  1.7  1.8  1.9  ...  1.15739  1.15740  \
0  1    1    1    1    1    1    1    1    1    1  ...        1        1   
1  1    1    1    1    1    1    1    1    1    1  ...        1        1   
2  1    1    1    1    1    1    1    1    1    1  ...        1        1   
3  1    1    1    1    1    1    1    1    1    1  ...        1        1   
4  1    1    1    1    1    1    1    1    1    1  ...        1        1   

   1.15741  1.15742  1.15743  1.15744  1.15745  1.15746  1.15747  0.636  
0        1        1        1        1        1        1        1      0  
1        1        1        1        1        1        1        1      0  
2        1        1        1        1        1        1        1      0  
3        1        1        1        1        1        1        1      0  
4        1        1        1        1     

In [ ]:
# Define la ruta de salida para el mejor modelo
ruta_modelo_salida = os.path.join(output_folder, 'B88639_Alex_Víquez.joblib')

# Separa las características (X) y la etiqueta objetivo (y)
# Asumiendo que la última columna es la etiqueta y el resto son características
X = datasetgrupal_total.iloc[:, :-1]
y = datasetgrupal_total.iloc[:, -1]

print("\nDistribución de clases en el conjunto de datos completo:")
print(y.value_counts().sort_index())

# Divide los datos en 80% para entrenamiento y 20% para prueba
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y  # Asegura que la distribución de clases sea similar en los conjuntos de entrenamiento y prueba
)

print("\nDistribución de clases en el conjunto de entrenamiento:")
print(y_train.value_counts().sort_index())

print("\nDistribución de clases en el conjunto de prueba:")
print(y_test.value_counts().sort_index())

print(f"\nNúmero de muestras de entrenamiento: {len(X_train)}")
print(f"Número de muestras de prueba: {len(X_test)}")


Distribución de clases en el conjunto de datos completo:
0.636
0    281
1    264
Name: count, dtype: int64

Distribución de clases en el conjunto de entrenamiento:
0.636
0    225
1    211
Name: count, dtype: int64

Distribución de clases en el conjunto de prueba:
0.636
0    56
1    53
Name: count, dtype: int64

Número de muestras de entrenamiento: 436
Número de muestras de prueba: 109




Definiremos un conjunto de modelos de aprendizaje para evaluar, incluyendo Máquinas de Vectores de Soporte (SVM), K-Vecinos más Cercanos (KNN), Naive Bayes, Árboles de Decisión y Bosques Aleatorios. Para algunos modelos, `StandardScaler` se utiliza dentro de un `Pipeline` para normalizar los datos antes del entrenamiento.

In [ ]:
# Paso 4: Definir Modelos de Aprendizaje
# Diccionario de modelos a evaluar
modelos = {
    "SVM Lineal": Pipeline([
        ("scaler", StandardScaler()),
        ("modelo", SVC(kernel='linear', random_state=42))
    ]),
    "SVM RBF": Pipeline([
        ("scaler", StandardScaler()),
        ("modelo", SVC(kernel='rbf', random_state=42))
    ]),
    "KNN k=3": Pipeline([
        ("scaler", StandardScaler()),
        ("modelo", KNeighborsClassifier(n_neighbors=3))
    ]),
    "KNN k=5": Pipeline([
        ("scaler", StandardScaler()),
        ("modelo", KNeighborsClassifier(n_neighbors=5))
    ]),
    "KNN k=7": Pipeline([
        ("scaler", StandardScaler()),
        ("modelo", KNeighborsClassifier(n_neighbors=7))
    ]),
    "Naive Bayes": GaussianNB(),
    "Árbol de Decisión": DecisionTreeClassifier(
        random_state=42
    ),
    "Árbol de Decisión max_depth=5": DecisionTreeClassifier(
        random_state=42,
        max_depth=5
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42
    )
}


Cada modelo definido será entrenado utilizando los datos de entrenamiento y evaluado en los datos de prueba. Se informará la puntuación de precisión para cada modelo.

In [ ]:
# Paso 5: Entrenar y Evaluar Modelos
print("\n--- Entrenando modelos y evaluando la precisión ---\n")

resultados = {}

for nombre, modelo in modelos.items():
    # Entrenar el modelo
    modelo.fit(X_train, y_train)

    # Realizar predicciones sobre el conjunto de prueba
    predicciones = modelo.predict(X_test)

    # Calcular la precisión
    acc = accuracy_score(y_test, predicciones)

    # Almacenar resultados
    resultados[nombre] = {
        "accuracy": acc,
        "modelo": modelo,
        "predicciones": predicciones
    }

    print(f"{nombre:<35} | Accuracy: {acc:.4f}")


--- Entrenando modelos y evaluando la precisión ---

SVM Lineal                          | Accuracy: 1.0000
SVM RBF                             | Accuracy: 0.8349
KNN k=3                             | Accuracy: 0.9083
KNN k=5                             | Accuracy: 0.7064
KNN k=7                             | Accuracy: 0.8716
Naive Bayes                         | Accuracy: 0.7706
Árbol de Decisión                   | Accuracy: 0.9908
Árbol de Decisión max_depth=5       | Accuracy: 0.6606
Random Forest                       | Accuracy: 0.9450




Después de evaluar todos los modelos, el que tenga la mayor precisión será seleccionado como el mejor modelo. Se mostrará su matriz de confusión y el informe de clasificación, y el modelo se guardará en Google Drive usando `joblib` para uso futuro.

In [ ]:
# Paso 6: Seleccionar y Guardar el Mejor Modelo

# Encontrar el mejor modelo basado en la precisión
mejor_nombre = max(resultados, key=lambda x: resultados[x]["accuracy"])
mejor_acc = resultados[mejor_nombre]["accuracy"]
mejor_modelo = resultados[mejor_nombre]["modelo"]
mejores_predicciones = resultados[mejor_nombre]["predicciones"]

print("\n" + "-" * 55)
print(f"Mejor modelo: {mejor_nombre}")
print(f"Precisión obtenida: {mejor_acc:.4f}")

print("\nMatriz de confusión para el mejor modelo:")
print(confusion_matrix(y_test, mejores_predicciones))

print("\nInforme de clasificación para el mejor modelo:")
print(classification_report(y_test, mejores_predicciones))

# Guardar el mejor modelo en Google Drive
joblib.dump(mejor_modelo, ruta_modelo_salida)

print("-" * 55)
print("Mejor modelo guardado en:")
print(ruta_modelo_salida)


-------------------------------------------------------
Mejor modelo: SVM Lineal
Precisión obtenida: 1.0000

Matriz de confusión para el mejor modelo:
[[56  0]
 [ 0 53]]

Informe de clasificación para el mejor modelo:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        56
           1       1.00      1.00      1.00        53

    accuracy                           1.00       109
   macro avg       1.00      1.00      1.00       109
weighted avg       1.00      1.00      1.00       109

-------------------------------------------------------
Mejor modelo guardado en:
/content/drive/MyDrive/FotosIAproyecto/Modelo/B88639_Alex_Víquez.joblib
